In [0]:
%pip install prophet
%pip install statsmodels

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, acf, pacf
import logging
import warnings
warnings.filterwarnings('ignore')
logging.getLogger('prophet').setLevel(logging.CRITICAL)

In [0]:
# Aplicando paleta de cores
COLORS = {
    'primary': '#1e3a5f',      
    'secondary': '#20b2aa',     
    'accent': '#00ff7f',        
    'success': '#32cd32',       
    'warning': '#ffa500',    
    'danger': '#ff6b6b',     
    'info': '#4169e1',        
    'light': '#f8f9fa',       
    'dark': '#2c3e50',        
    'gradient_start': '#1e3a5f',
    'gradient_end': '#20b2aa'
}

plt.style.use('default')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': COLORS['dark'],
    'axes.linewidth': 1.2,
    'axes.labelcolor': COLORS['dark'],
    'text.color': COLORS['dark'],
    'xtick.color': COLORS['dark'],
    'ytick.color': COLORS['dark'],
    'grid.color': '#e0e0e0',
    'grid.alpha': 0.6,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 10
})


In [0]:
query = """
SELECT 
    fs.order_date,
    fs.product_fk,
    fs.territory_fk,
    fs.order_quantity,
    fs.unit_price,
    fs.total_due,
    dp.product_name,
    dp.product_category_name,
    dt.country_region_name,
    ds.store_name,
    ds.business_entity_id as store_id
FROM ted_dev.marts.fact_sales fs
JOIN ted_dev.marts.dim_product dp ON fs.product_fk = dp.product_pk
JOIN ted_dev.marts.dim_territory dt ON fs.territory_fk = dt.territory_pk
LEFT JOIN ted_dev.marts.dim_store ds ON fs.sales_person_fk = ds.sales_person_id
ORDER BY fs.order_date
"""

df = spark.sql(query).toPandas()
df['order_date'] = pd.to_datetime(df['order_date'])

print(f"Dados carregados: {len(df):,} registros")
print(f"Período: {df['order_date'].min()} a {df['order_date'].max()}")
print(f"Produtos únicos: {df['product_fk'].nunique()}")
print(f"Lojas distintas: {df['store_id'].nunique()}")

monthly_data = df.groupby([
    'product_fk', 
    'store_id',
    pd.Grouper(key='order_date', freq='M')
]).agg({
    'order_quantity': 'sum',
    'unit_price': 'mean',
    'total_due': 'sum',
    'product_name': 'first',
    'store_name': 'first',
    'country_region_name': 'first'
}).reset_index()

monthly_data = monthly_data.dropna(subset=['store_id'])

In [0]:
display(monthly_data)

# Previsão de demanda por produto/loja

## Objetivos

Foi desenvolvido uma comparação entre diferentes abordagens de previsão de demanda para otimizar o planejamento de compras e distribuição, utilizando médias móveis como baseline de referência.

In [0]:
fig, axes = plt.subplots(2, 3, figsize=(24, 12))
all_items = [(top_3_products, "Top 3 Produtos"), (top_3_stores, "Top 3 Lojas")]

for row in range(2):
    items, section_title = all_items[row]
    
    for col in range(3):
        ax = axes[row, col]
        
        if col < len(items):
            name, ((product, store), forecast_data) = items[col]
            historical = forecast_data['historical_data']
            
            dates = historical['order_date']
            quantities = historical['order_quantity']
            ma = quantities.rolling(window=3, center=False).mean()
            
            last_date = dates.iloc[-1]
            future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=3, freq='M')
            forecast_values = forecast_data['forecast_3_months']
            
            ci = confidence_intervals[(product, store)]
            
            ax.plot(dates, quantities, 'o-', color=COLORS['primary'], 
                    label='Histórico', alpha=0.8, markersize=4, linewidth=2)
            
            ax.plot(dates, ma, '--', color=COLORS['secondary'], 
                    label='Média Móvel (3)', alpha=0.9, linewidth=2.5)
            
            ax.plot(future_dates, forecast_values, 's-', color=COLORS['warning'], 
                    label='Previsão', markersize=7, linewidth=3, alpha=0.9)
            
            ax.fill_between(future_dates, 
                            [ci['lower_95']] * 3,
                            [ci['upper_95']] * 3,
                            alpha=0.25, color=COLORS['warning'], label='IC 95%')
            
            ax.axvline(x=last_date, color=COLORS['danger'], linestyle=':', alpha=0.6, linewidth=1.5)
            
            title = name if row == 0 else forecast_data['store_name']
            ax.set_title(title, fontweight='bold', fontsize=11, pad=15)
            
            if col == 0:
                ax.legend(loc='best', framealpha=0.9, fontsize=9, frameon=True)
            
            ax.grid(True, alpha=0.3)
            ax.tick_params(axis='x', rotation=45, labelsize=9)
            ax.tick_params(axis='y', labelsize=9)
            
            forecast_mean = forecast_data['forecast_mean']
            ax.text(0.02, 0.98, f'Prev: {forecast_mean:.1f} un/mês', 
                    transform=ax.transAxes, verticalalignment='top', fontsize=9,
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
        else:
            ax.set_visible(False)

fig.suptitle('Previsões - Top 3 Produtos vs Top 3 Lojas', 
             fontsize=18, fontweight='bold', y=0.98)

axes[0, 1].text(0.5, 1.15, 'Top 3 Produtos', transform=axes[0, 1].transAxes, 
                ha='center', va='bottom', fontsize=14, fontweight='bold')
axes[1, 1].text(0.5, 1.15, 'Top 3 Lojas', transform=axes[1, 1].transAxes, 
                ha='center', va='bottom', fontsize=14, fontweight='bold')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


In [0]:
def calculate_all_metrics(actual, predicted):
    """Calcula MAE, RMSE e MAPE para avaliação do modelo."""
    actual, predicted = np.array(actual), np.array(predicted)
    mae = np.mean(np.abs(actual - predicted))
    rmse = np.sqrt(np.mean((actual - predicted)**2))
    mape = np.mean(np.abs((actual - predicted) / (actual + 1e-8))) * 100
    return mae, rmse, mape

def fit_arima(train_data, test_size=3):
    """Ajusta um modelo ARIMA"""
    try:
        is_stationary = adfuller(train_data)[1] < 0.05
        d = 0 if is_stationary else 1
        best_aic, best_model = float('inf'), None
        for p in [0, 1, 2]:
            for q in [0, 1, 2]:
                try:
                    model = ARIMA(train_data, order=(p, d, q))
                    fitted = model.fit()
                    if fitted.aic < best_aic:
                        best_aic, best_model = fitted.aic, fitted
                except:
                    continue
        if best_model:
            return np.maximum(best_model.forecast(steps=test_size), 0)
        return None
    except Exception as e:
        print(f"Erro ao ajustar modelo: {e}")
        return None

def fit_prophet(train_data, test_size=3):
    """Ajusta um modelo Prophet"""
    try:
        df_prophet = pd.DataFrame({'ds': pd.to_datetime(train_data.index), 'y': train_data.values})
        model = Prophet(
            yearly_seasonality='auto', weekly_seasonality=False, daily_seasonality=False,
            changepoint_prior_scale=0.05, seasonality_mode='additive'
        )
        model.fit(df_prophet)
        future = model.make_future_dataframe(periods=test_size, freq='M')
        forecast = model.predict(future)
        return np.maximum(forecast.tail(test_size)['yhat'].values, 0)
    except Exception as e:
        print(f"Erro ao ajustar modelo: {e}")
        return None

baseline_forecasts = {}
for (product, store), group in monthly_data.groupby(['product_fk', 'store_id']):
    if len(group) >= 3:
        group = group.sort_values('order_date')
        last_ma = group['order_quantity'].rolling(window=3).mean().iloc[-1]
        if not pd.isna(last_ma):
            baseline_forecasts[(product, store)] = {
                'product_name': group['product_name'].iloc[0],
                'forecast_mean': last_ma,
                'historical_data': group.set_index('order_date')[['order_quantity']].copy()
            }

top_products_candidates = {}
for (product, store), data in baseline_forecasts.items():
    if data['product_name'] not in top_products_candidates or data['forecast_mean'] > top_products_candidates[data['product_name']][1]['forecast_mean']:
        top_products_candidates[data['product_name']] = ((product, store), data)

top_3_products_to_validate = sorted(top_products_candidates.items(), key=lambda item: item[1][1]['forecast_mean'], reverse=True)[:3]

print("Top 3 produtos selecionados para validação:")
for name, (keys, data) in top_3_products_to_validate:
    print(f"- {name} (Previsão de Baseline: {data['forecast_mean']:.2f} un/mês)")

results = []

for product_name, (keys, data) in top_3_products_to_validate:
    print(f"\nProduto: {data['product_name']}")
    hist_data = data['historical_data'].sort_index()
    
    if len(hist_data) >= 8:
        train, test = hist_data[:-3], hist_data[-3:]
        print(f"Dados: {len(train)} treino | {len(test)} teste")
        
        test_actual = test['order_quantity'].values
        train_values = train['order_quantity']
        
        baseline_pred = [train_values.rolling(window=3).mean().iloc[-1]] * 3
        arima_pred = fit_arima(train_values, test_size=3)
        prophet_pred = fit_prophet(train_values, test_size=3)
        
        models = {'Baseline (MA3)': baseline_pred}
        if arima_pred is not None: models['ARIMA'] = arima_pred.tolist()
        if prophet_pred is not None: models['Prophet'] = prophet_pred.tolist()
        
        model_results = {}
        print("Modelo             | MAE    | RMSE   | MAPE(%)")
        print("-------------------|--------|--------|--------")
        for name, pred in models.items():
            mae, rmse, mape = calculate_all_metrics(test_actual, pred)
            model_results[name] = {'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'pred': pred}
            print(f"{name:18} | {mae:6.1f} | {rmse:6.1f} | {mape:6.1f}")
        
        if model_results:
            best = min(model_results.items(), key=lambda x: x[1]['MAE'])
            print(f"Melhor: {best[0]} (MAE: {best[1]['MAE']:.1f})")
            results.append({
                'product_name': data['product_name'], 'train': train, 'test': test,
                'test_actual': test_actual, 'models': model_results, 'best': best[0]
            })
    else:
        print(f"Dados insuficientes para {data['product_name']}")

In [0]:
model_colors = {
    'Histórico (Treino)': COLORS['dark'],
    'Real (Teste)': COLORS['primary'],
    'Baseline (MA3)': COLORS['warning'],
    'ARIMA': COLORS['secondary'],
    'Prophet': COLORS['info']
}

if results:   
    num_plots = len(results)
    fig, axes = plt.subplots(num_plots, 1, figsize=(15, 6 * num_plots), squeeze=False, facecolor='white')
    axes = axes.flatten()

    for i, result in enumerate(results):
        ax = axes[i]
        train_df, test_df = result['train'], result['test']

        ax.plot(train_df.index, train_df['order_quantity'], 
                color=model_colors.get('Histórico (Treino)', 'gray'), 
                marker='.', linestyle='-', alpha=0.6, label='Histórico (Treino)')

        ax.plot(test_df.index, result['test_actual'], 'o-', 
                color=model_colors.get('Real (Teste)', 'black'), 
                markersize=8, linewidth=2.5, label='Real (Teste)')

        for name, data in result['models'].items():
            ax.plot(test_df.index, data['pred'], linestyle='--', marker='s', markersize=6, 
                    color=model_colors.get(name, 'gray'), 
                    label=f'{name} (MAE: {data["MAE"]:.1f})')
        
        ax.axvline(train_df.index[-1], color=COLORS['danger'], linestyle='--', linewidth=1.5, alpha=0.8)
        
        ax.set_title(result['product_name'], fontsize=16, weight='bold', pad=40)  
        
        best_name, best_mae = result['best'], result['models'][result['best']]['MAE']
        ax.text(0.5, 1.08, f"Melhor modelo: {best_name} (MAE: {best_mae:.2f})", 
                transform=ax.transAxes, ha='center', fontsize=12, style='italic') 
        
        ax.set_ylabel('Quantidade vendida', fontsize=12)
        ax.legend(loc='upper left', fontsize=10)
        ax.tick_params(axis='x', rotation=15)
        
    fig.suptitle('Comparativo de performance dos modelos de previsão nos top 3 produtos', 
                fontsize=20, weight='bold', y=0.95) 
    
    plt.tight_layout(rect=[0, 0, 1, 0.93], h_pad=4.0) 
    plt.show()